In [6]:
from pathlib import Path
import json
import pandas as pd

try:
    from IPython.display import display  # type: ignore
except Exception:
    # Fallback for non-notebook execution
    display = print

# Reload local notebook utilities to pick up edits without restarting kernel
import importlib
import testLibs as tl
importlib.reload(tl)

# Notebook helpers (scoring + flatteners)
ResultsFlattener = tl.ResultsFlattener
mznResultsFlattener = tl.mznResultsFlattener
get_significative_solvers = tl.get_significative_solvers
scoreComputation_subset = tl.scoreComputation_subset
compute_llm_scores = tl.compute_llm_scores
compute_top1_llm_scores = tl.compute_top1_llm_scores
compute_closed_gap = tl.compute_closed_gap
build_llm_performance_table = tl.build_llm_performance_table
filter_to_solvers = tl.filter_to_solvers
singleSolverScore = tl.singleSolverScore
scoreComputation = tl.scoreComputation

# Optional plotting libs (only needed for plots)
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None
try:
    import seaborn as sns
except ModuleNotFoundError:
    sns = None

In [7]:
data_dir = Path("../data/testOutputNameless").resolve()
if not data_dir.is_dir():
    raise FileNotFoundError(f"Expected folder not found: {data_dir}")

json_paths = sorted(data_dir.glob("*.json"))
print(f"Loading {len(json_paths)} JSON files from {data_dir}")

data_by_file = {}
for p in json_paths:
    with p.open("r", encoding="utf-8") as f:
        data_by_file[p.name] = json.load(f)

# List in a stable order, matching json_paths
data_list = [data_by_file[p.name] for p in json_paths]

print("Loaded files:")
for name in data_by_file.keys():
    print(" -", name)

with open('../data/tablesJSON/allTables_free.json', 'r') as f1:
    MznResults = json.load(f1)

Loading 5 JSON files from /home/vro5/Coding/AgenticSolvers/test/data/testOutputNameless
Loaded files:
 - LLMsuggestions_nameless_chat_fzn_Sdesc_T0p2.json
 - LLMsuggestions_nameless_chat_fzn_Sdesc_T0p7.json
 - LLMsuggestions_nameless_featOnly_Sdesc_T0p3.json
 - LLMsuggestions_nameless_featOnly_Sdesc_T0p7.json
 - LLMsuggestions_nameless_uncommented_desc_solverdesc.json


In [8]:
# --- Full free-solvers analysis (parallel, single, closed-gap) ---
# Flatten MiniZinc results and compute scores over the full free-solver set
mzn_raw_df = mznResultsFlattener(MznResults)
scored_df = scoreComputation(mzn_raw_df)
print(f"Computed solver scores: {len(scored_df)} rows")



Computed solver scores: 2000 rows


In [9]:
# --- Reconduction: remap nameless suggestions (a,b,c,...) to real solvers and re-score ---
from pathlib import Path
import json
import pandas as pd

# Load nameless solvers mapping (local test data) -- maps letters -> solver ids/descriptions
nameless_path = Path("../data/namelessSolvers.json").resolve()
if not nameless_path.is_file():
    raise FileNotFoundError(f"Expected mapping file not found: {nameless_path}")
with nameless_path.open("r", encoding="utf-8") as nf:
    nameless_data = json.load(nf)
# Build remap: letter (e.g., 'a') -> solver identifier (from 'correspondences')
remap = nameless_data.get('correspondences', {}) if isinstance(nameless_data, dict) else {}
print(f"Loaded remap for {len(remap)} nameless entries")

def _remap_top3_list(val):
    if val is None:
        return None
    if isinstance(val, list):
        return [remap.get(str(v).strip(), str(v).strip()) for v in val]
    # handle string-encoded lists
    parts = [p.strip() for p in str(val).replace(';', ',').split(',') if p.strip()]
    return [remap.get(p, p) for p in parts] if parts else None

def _remap_top1(val):
    if val is None:
        return None
    s = str(val).strip()
    return remap.get(s, s)

# Re-score each input file after remapping suggestions
recon_top3_parts = []
recon_top1_parts = []
recon_cg_parts = []
recon_tables_by_file = {}

for fname, llm_results in data_by_file.items():
    llm_df = ResultsFlattener(llm_results)
    if llm_df is None or llm_df.empty:
        continue

    # create remapped copy
    df2 = llm_df.copy()
    if 'top3_list' in df2.columns:
        df2['top3_list'] = df2['top3_list'].apply(_remap_top3_list)
    if 'top1' in df2.columns:
        df2['top1'] = df2['top1'].apply(_remap_top1)

    # compute scores using full scored_df (free solvers)
    top3_summary = compute_llm_scores(df2, scored_df)
    top1_summary, top1_scored = compute_top1_llm_scores(df2, scored_df)
    cg_rows = compute_closed_gap(top1_scored, scored_df, allowed_solvers=None, sbs_solver=None)
    cg_df = pd.DataFrame(cg_rows) if cg_rows else pd.DataFrame()
    if 'ClosedGap' not in cg_df.columns:
        for c in ['provider', 'model', 'InstancesCovered', 'AS', 'SBS', 'VBS', 'ClosedGap']:
            if c not in cg_df.columns:
                cg_df[c] = pd.NA

    perf_table = build_llm_performance_table(
        top3_summary=top3_summary,
        top1_summary=top1_summary,
        closed_gap=cg_df,
        sort_by='SingleScore',
        ascending=False,
    )

    recon_tables_by_file[fname] = {
        'top3_summary': top3_summary,
        'top1_summary': top1_summary,
        'closed_gap': cg_df,
        'performance_table': perf_table,
    }

    # Collect parts for aggregated views
    if top3_summary is not None and not top3_summary.empty:
        t3 = top3_summary.drop(columns=['provider'], errors='ignore')
        t3['source_file'] = fname
        recon_top3_parts.append(t3)
    if top1_summary is not None and not top1_summary.empty:
        t1 = top1_summary.drop(columns=['provider'], errors='ignore')
        t1['source_file'] = fname
        recon_top1_parts.append(t1)
    if not cg_df.empty:
        cg_part = cg_df.drop(columns=['provider'], errors='ignore')
        cg_part['source_file'] = fname
        recon_cg_parts.append(cg_part)

print(f"Re-scored {len(recon_tables_by_file)} files after remapping nameless suggestions")

recon_top3_all = pd.concat(recon_top3_parts, ignore_index=True) if recon_top3_parts else pd.DataFrame()
recon_top1_all = pd.concat(recon_top1_parts, ignore_index=True) if recon_top1_parts else pd.DataFrame()
recon_cg_all = pd.concat(recon_cg_parts, ignore_index=True) if recon_cg_parts else pd.DataFrame()

# Display per-file performance tables (stable order)
for p in json_paths:
    fname = p.name
    if fname not in recon_tables_by_file:
        continue
    print("=" * 80)
    print("Reconduced File:", fname)
    print("=" * 80)
    print("Performance table:")
    display(recon_tables_by_file[fname]['performance_table'])

# Aggregated summary per file: best Single, Parallel, ClosedGap after reconduction
def _best_by_file(df: pd.DataFrame, value_col: str) -> pd.DataFrame:
    if df is None or df.empty or value_col not in df.columns:
        return pd.DataFrame(columns=['source_file', value_col])
    out = df.dropna(subset=['source_file']).copy()
    out[value_col] = pd.to_numeric(out[value_col], errors='coerce')
    out = out.dropna(subset=[value_col])
    if out.empty:
        return pd.DataFrame(columns=['source_file', value_col])
    out = (
        out.sort_values(['source_file', value_col], ascending=[True, False])
        .groupby(['source_file'], as_index=False)
        .head(1)
        .reset_index(drop=True)
    )
    return out[['source_file', value_col]].copy()

single_best = _best_by_file(recon_top1_all, 'LLM_Top1_TotalScore')
parallel_best = _best_by_file(recon_top3_all, 'LLM_TotalScore')
cg_best = _best_by_file(recon_cg_all, 'ClosedGap')

summary_recon = single_best.merge(parallel_best, on='source_file', how='outer')
summary_recon = summary_recon.merge(cg_best, on='source_file', how='outer')
if summary_recon.empty:
    display("No reconduced summary rows available")
else:
    summary_recon = summary_recon.rename(columns={
        'source_file': 'File',
        'LLM_Top1_TotalScore': 'Single Score',
        'LLM_TotalScore': 'Parallel Score',
        'ClosedGap': 'Closed Gap'
    })
    summary_recon = summary_recon[['File', 'Single Score', 'Parallel Score', 'Closed Gap']]
    summary_recon['Single Score'] = pd.to_numeric(summary_recon['Single Score'], errors='coerce')
    summary_recon['Parallel Score'] = pd.to_numeric(summary_recon['Parallel Score'], errors='coerce')
    summary_recon['Closed Gap'] = pd.to_numeric(summary_recon['Closed Gap'], errors='coerce')

    # Map filenames -> Variant labels (reuse simple rules from swappedAnalysis)
    import re
    def _map_variant_from_filename(fname: str) -> str:
        if fname is None:
            return ''
        fn = str(fname)
        if 'featOnly_Sdesc' in fn or 'featOnly_Pdesc_Sdesc' in fn:
            return 'Features + Solvers Description'
        if 'featOnly_Pdesc_Sdesc' in fn or 'featOnly_Pdesc' in fn:
            return 'Features + Problem Description'
        if 'uncommented_desc' in fn or 'uncommented' in fn or 'uncommented_desc_solverdesc' in fn:
            return 'Scripts + Problem Description + Solvers Description'
        if 'fzn_Sdesc' in fn or 'chat_fzn_Sdesc' in fn or fn.startswith('LLMsuggestions_chat_fzn'):
            return 'fzn2nl + Solver Description'
        return fn

    def _extract_temp_from_filename(fname: str) -> str:
        if fname is None:
            return ''
        m = re.search(r"(T\d+(?:p\d+)?)", str(fname))
        return m.group(1) if m else ''

    summary_recon['Variant'] = summary_recon['File'].apply(_map_variant_from_filename)
    summary_recon['Temperature'] = summary_recon['File'].apply(_extract_temp_from_filename)

    # Reorder to include Variant and Temperature, drop File
    summary_recon = summary_recon[['Variant', 'Temperature', 'Single Score', 'Parallel Score', 'Closed Gap']]

    summary_recon = summary_recon.sort_values(['Single Score', 'Parallel Score'], ascending=[False, False], na_position='last').reset_index(drop=True)
    display(summary_recon.style.hide(axis="index"))


Loaded remap for 20 nameless entries
Re-scored 5 files after remapping nameless suggestions
Reconduced File: LLMsuggestions_nameless_chat_fzn_Sdesc_T0p2.json
Performance table:


,Model,Single Score,Parallel Score,Closed Gap
0,openai/gpt-oss-120b,10.129255,56.08183,-5.553108


Reconduced File: LLMsuggestions_nameless_chat_fzn_Sdesc_T0p7.json
Performance table:


,Model,Single Score,Parallel Score,Closed Gap
0,openai/gpt-oss-120b,4.022972,55.112768,-6.060458


Reconduced File: LLMsuggestions_nameless_featOnly_Sdesc_T0p3.json
Performance table:


...

Reconduced File: LLMsuggestions_nameless_featOnly_Sdesc_T0p7.json
Performance table:


,Model,Single Score,Parallel Score,Closed Gap
0,openai/gpt-oss-120b,9.103129,63.225334,-5.638365


Reconduced File: LLMsuggestions_nameless_uncommented_desc_solverdesc.json
Performance table:


,Model,Single Score,Parallel Score,Closed Gap
0,openai/gpt-oss-120b,5.816595,34.567432,-5.911432


Variant,Temperature,Single Score,Parallel Score,Closed Gap
fzn2nl + Solver Description,T0p2,10.129255,56.081830,-5.553108
Features + Solvers Description,T0p7,9.103129,63.225334,-5.638365
Features + Solvers Description,T0p3,9.058760,64.352661,-5.642051
Scripts + Problem Description + Solvers Description,,5.816595,34.567432,-5.911432
fzn2nl + Solver Description,T0p7,4.022972,55.112768,-6.060458


In [10]:
# --- Suggestion occurrence analysis per file, grouped per Variant+Temperature ---
from collections import Counter, defaultdict
import re

# Ensure remap (letter -> solver id) is available (created in the reconduction cell above)
if 'remap' not in globals():
    nameless_path = Path("../data/namelessSolvers.json").resolve()
    if nameless_path.is_file():
        with nameless_path.open("r", encoding="utf-8") as _nf:
            try:
                _nd = json.load(_nf)
                remap = _nd.get('correspondences', {}) if isinstance(_nd, dict) else {}
            except Exception:
                remap = {}
    else:
        remap = {}

# Helper: map filename -> Variant and extract Temperature token (same rules used elsewhere)
def _map_variant_from_filename(fname: str) -> str:
    if fname is None:
        return ''
    fn = str(fname)
    if 'featOnly_Sdesc' in fn or 'featOnly_Pdesc_Sdesc' in fn:
        return 'Features + Solvers Description'
    if 'featOnly_Pdesc_Sdesc' in fn or 'featOnly_Pdesc' in fn:
        return 'Features + Problem Description'
    if 'uncommented_desc' in fn or 'uncommented' in fn or 'uncommented_desc_solverdesc' in fn:
        return 'Scripts + Problem Description + Solvers Description'
    if 'fzn_Sdesc' in fn or 'chat_fzn_Sdesc' in fn or fn.startswith('LLMsuggestions_chat_fzn'):
        return 'fzn2nl + Solver Description'
    return 'Other'


def _extract_temp_from_filename(fname: str) -> str:
    if fname is None:
        return ''
    m = re.search(r'(T\d+(?:p\d+)?)', str(fname))
    return m.group(1) if m else ''

occurrence_tables_by_group = defaultdict(list)

for p in json_paths:
    fname = p.name
    llm_results = data_by_file.get(fname)
    if llm_results is None:
        continue
    df = ResultsFlattener(llm_results)
    if df is None or df.empty:
        continue

    top1_counter = Counter()
    total_counter = Counter()

    # Count occurrences per instance but ensure TotalCount counts unique instances
    for _, row in df.iterrows():
        instance_seen = set()
        # top1
        if 'top1' in df.columns:
            v = row.get('top1')
            if v is not None and pd.notna(v):
                s = str(v).strip()
                if s:
                    top1_counter[s] += 1
                    instance_seen.add(s)
        # top3_list (parallel suggestions)
        if 'top3_list' in df.columns:
            lst = row.get('top3_list')
            if isinstance(lst, list):
                for it in lst:
                    if it is None:
                        continue
                    s = str(it).strip()
                    if s:
                        instance_seen.add(s)
            else:
                parts = [pp.strip() for pp in str(lst).replace(';', ',').split(',') if pp.strip()]
                for it in parts:
                    instance_seen.add(it)
        # increment total_counter once per solver for this instance
        for s in instance_seen:
            total_counter[s] += 1

    # Build per-file table rows
    names = sorted(set(list(total_counter.keys()) + list(top1_counter.keys())))
    occ_rows = []
    for name in names:
        orig = remap.get(name, pd.NA)
        occ_rows.append({
            'SuggestedName': name,
            'OriginalSolver': orig,
            'Top1Count': int(top1_counter.get(name, 0)),
            'TotalCount': int(total_counter.get(name, 0)),
            'File': fname,
        })
    if not occ_rows:
        continue
    occ_df = pd.DataFrame(occ_rows)
    occ_df['Variant'] = occ_df['File'].apply(_map_variant_from_filename)
    occ_df['Temperature'] = occ_df['File'].apply(_extract_temp_from_filename)

    key = (occ_df.iloc[0]['Variant'], occ_df.iloc[0]['Temperature'])
    occurrence_tables_by_group[key].append(occ_df)

# Display one table per Variant+Temperature (concatenate per-files within group)
for (variant, temp), tables in sorted(occurrence_tables_by_group.items()):
    header = f"Suggestion counts — Variant: {variant} / Temp: {temp if temp else 'unknown'}"
    print('\n' + '=' * 80)
    print(header)
    print('=' * 80)
    group_df = pd.concat(tables, ignore_index=True)
    # Show per-file counts, sorted by Top1Count then TotalCount
    group_df = group_df.sort_values(['Top1Count', 'TotalCount'], ascending=[False, False]).reset_index(drop=True)
    display(group_df[['SuggestedName', 'OriginalSolver', 'Top1Count', 'TotalCount']].style.hide(axis="index"))



Suggestion counts — Variant: Features + Solvers Description / Temp: T0p3


SuggestedName,OriginalSolver,Top1Count,TotalCount
a,atlantis-free,75,78
c,choco-solver__cp_-free,18,96
b,cbc-free,6,80
d,choco-solver__cp_-par,1,4
g,cp_optimizer-free,0,19
i,gurobi-free,0,14
o,or-tools_cp-sat_ls-free,0,5
p,picatsat-free,0,2
e,choco-solver__cp-sat_-free,0,1
f,chuffed-free,0,1



Suggestion counts — Variant: Features + Solvers Description / Temp: T0p7


SuggestedName,OriginalSolver,Top1Count,TotalCount
a,atlantis-free,83,85
c,choco-solver__cp_-free,13,96
b,cbc-free,3,86
d,choco-solver__cp_-par,1,4
g,cp_optimizer-free,0,13
i,gurobi-free,0,11
e,choco-solver__cp-sat_-free,0,2
j,highs-free,0,2
f,chuffed-free,0,1



Suggestion counts — Variant: Scripts + Problem Description + Solvers Description / Temp: unknown


SuggestedName,OriginalSolver,Top1Count,TotalCount
a,atlantis-free,49,57
Chuffed,,30,30
b,cbc-free,9,59
chuffed,,5,5
c,choco-solver__cp_-free,4,54
t,yuck-free,2,2
Gecode,,1,29
OR-Tools,,0,16
OR‑Tools,,0,6
d,choco-solver__cp_-par,0,5



Suggestion counts — Variant: fzn2nl + Solver Description / Temp: T0p2


SuggestedName,OriginalSolver,Top1Count,TotalCount
a,atlantis-free,81,81
c,choco-solver__cp_-free,11,94
Chuffed,,6,6
b,cbc-free,2,83
g,cp_optimizer-free,0,11
o,or-tools_cp-sat_ls-free,0,10
Gecode,,0,6
OR-Tools,,0,4
d,choco-solver__cp_-par,0,2
OR-Tools CP-SAT,,0,1



Suggestion counts — Variant: fzn2nl + Solver Description / Temp: T0p7


SuggestedName,OriginalSolver,Top1Count,TotalCount
a,atlantis-free,83,83
c,choco-solver__cp_-free,8,91
Chuffed,,5,5
chuffed,,1,1
b,cbc-free,0,83
g,cp_optimizer-free,0,8
o,or-tools_cp-sat_ls-free,0,7
Gecode,,0,5
OR-Tools,,0,5
gecode,,0,1
